# Relative quality at twenty seeds

Section 3.4's relative-quality variant is the paper's proposed fix for
the PneumoniaMNIST failure: referencing each client against its own
first-round quality should cancel the task-difficulty term that the
absolute signal confounds with corruption.

At ten seeds on PneumoniaMNIST it already carries AUROC (0.921 against
0.896, p = 0.006) and accuracy (0.802 against 0.761, p = 0.027), but ECE
-- the metric the paper leads on -- sits at p = 0.065, just the wrong
side of the line. That is the one number that decides whether the paper
can say it diagnosed a failure *and* fixed it, or only diagnosed it.

Twenty seeds resolve it either way. If ECE reaches significance the fix
is established on the corpus where the method fails; if it does not, the
variant helps discrimination but not calibration, and the paper says
that instead. Both are reportable; p = 0.065 at n = 10 is not.

PathMNIST is included at twenty seeds too, because the variant must not
cost anything on the corpus where the absolute signal already works.


In [ ]:
import glob, os, shutil, sys
REPO='/kaggle/working/repo'
hits=glob.glob('/kaggle/input/**/paper/build_paper.py', recursive=True)
assert hits, 'attach the aha-fed-code dataset'
src=os.path.dirname(os.path.dirname(hits[0]))
if not os.path.isdir(os.path.join(REPO,'paper')):
    shutil.copytree(src, REPO, dirs_exist_ok=True)
os.chdir(REPO); sys.path.insert(0, REPO)
print('staged:', os.getcwd())


In [ ]:
!pip install -q medmnist

In [ ]:
import subprocess, sys
PROBE=('import torch;'
       'a=torch.randn(4,3,8,8,device="cuda");'
       'w=torch.randn(2,3,3,3,device="cuda");'
       'torch.nn.functional.conv2d(a,w).sum().item();'
       'print(torch.cuda.get_device_name(0), torch.__version__)')
r=subprocess.run([sys.executable,'-c',PROBE],capture_output=True,text=True)
print(r.stdout.strip() or r.stderr.strip()[-400:])
if r.returncode!=0:
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    'torch==2.4.1','torchvision==0.19.1',
                    '--index-url','https://download.pytorch.org/whl/cu118'],check=True)
    r=subprocess.run([sys.executable,'-c',PROBE],capture_output=True,text=True)
    print(r.stdout.strip() or r.stderr.strip()[-400:])
    if r.returncode!=0: raise SystemExit('no working torch/GPU combination')


In [ ]:
import traceback
from paper.run_real_experiments import run_one, OUT_DIR

# ten existing seeds plus ten new ones; run_one skips what already exists
SEEDS=[42,123,456,7,13,21,99,2024,31337,555,
       1,2,3,5,8,77,101,777,4242,90210]
PNEU={'dataset':'pneumoniamnist','num_classes':2,'samples_per_client':300}
PATH={'dataset':'pathmnist','num_classes':9,'samples_per_client':1000}

done=fail=0
for s in SEEDS:
    for rel,tag in [(True,'relq'),(False,'absq')]:
        try:
            run_one(OUT_DIR/f'pneuval_{tag}_s{s}.json', aggregator='adaptive',
                    alpha=1.0, beta=4.0, gamma=4.0, relative_quality=rel,
                    seed=s, num_rounds=20, eval_split='val', **PNEU)
            done+=1
        except Exception:
            fail+=1; print(f'FAILED pneuval_{tag}_s{s}'); traceback.print_exc()
print(f'PneumoniaMNIST done ({done} ok, {fail} failed)', flush=True)

for s in SEEDS:
    for rel,tag in [(True,'relq'),(False,'absq')]:
        try:
            run_one(OUT_DIR/f'pathval_{tag}_s{s}.json', aggregator='adaptive',
                    alpha=1.0, beta=4.0, gamma=4.0, relative_quality=rel,
                    seed=s, num_rounds=25, eval_split='val', **PATH)
            done+=1
        except Exception:
            fail+=1; print(f'FAILED pathval_{tag}_s{s}'); traceback.print_exc()
print(f'TOTAL {done} ok, {fail} failed', flush=True)


In [ ]:
import glob, json, re, statistics as st
from pathlib import Path
from scipy.stats import wilcoxon

def by_seed(pat, key):
    out={}
    for f in sorted(glob.glob(pat)):
        d=json.loads(Path(f).read_text())
        fin=d.get('final') or {}
        m=re.search(r'_s(\d+)$', Path(f).stem)
        if fin.get(key) is not None:
            out[m.group(1) if m else '42']=fin[key]
    return out

for corpus in ('pneuval','pathval'):
    print(f'\n=== {corpus} ===')
    n=0
    for key in ('ece','accuracy','auroc'):
        a=by_seed(f'runs/paper/real/{corpus}_relq_s*.json',key)
        b=by_seed(f'runs/paper/real/{corpus}_absq_s*.json',key)
        c=sorted(set(a)&set(b))
        if len(c)<2: continue
        n=len(c)
        x=[a[k] for k in c]; y=[b[k] for k in c]
        p=wilcoxon(x,y).pvalue
        better=sum(1 for i in range(len(c))
                   if (x[i]<y[i] if key=='ece' else x[i]>y[i]))
        print(f'  {key:9s} rel {st.mean(x):.4f}  abs {st.mean(y):.4f}  '
              f'p={p:.4f}  rel better {better}/{len(c)}')
    if n:
        print(f'  n={n}')


In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/relq20_results.zip runs/paper/real
!ls -lh /kaggle/working/relq20_results.zip
